In [ ]:
import pandas as pd

# Walmart Store Sales — Course-end Project 1

**Objective:** Build a baseline model to predict `Weekly_Sales` for 45 Walmart stores, include EDA, feature engineering, and evaluate using a weighted metric that places holiday weeks 5x weight.

**Contents:** Data load, EDA, preprocessing, feature engineering, baseline RandomForest model, weighted evaluation.

In [ ]:
# Additional imports for analysis
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

%matplotlib inline
sns.set(style="whitegrid")

In [ ]:
# Load data
df = pd.read_csv('Walmart_Store_sales.csv', parse_dates=['Date'])
print('Rows, cols:', df.shape)
df.head()

In [ ]:
# Compute total Weekly_Sales per store and show the max
store_totals = df.groupby('Store')['Weekly_Sales'].sum().sort_values(ascending=False)
max_store = int(store_totals.index[0])
max_total = float(store_totals.iloc[0])
print(f'Store with maximum total sales: {max_store} — Total: {max_total:.2f}')
store_totals.head(10)

# Compute per-store mean, std, coefficient of variation (std/mean) and mean-to-std
stats = df.groupby('Store')['Weekly_Sales'].agg(['mean','std']).rename(columns={'mean':'Mean','std':'Std'})
stats['CV'] = stats['Std'] / stats['Mean']
stats['Mean_to_Std'] = stats['Mean'] / stats['Std']
# Which store has maximum std
stats_sorted_std = stats.sort_values('Std', ascending=False)
max_std_store = int(stats_sorted_std.index[0])
print(f"Store with maximum std (most variable sales): {max_std_store} — Std: {stats_sorted_std.loc[max_std_store,'Std']:.2f}, Mean: {stats_sorted_std.loc[max_std_store,'Mean']:.2f}, CV: {stats_sorted_std.loc[max_std_store,'CV']:.4f}")
# Show top 10 by std
stats_sorted_std.head(10)

# Quarterly growth: Q2 2012 -> Q3 2012 per store
import pandas as pd
pd.options.display.float_format = '{:,.2f}'.format
# Ensure Date is datetime and extract Year/Month (if not already present)
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df2012 = df[df['Year'] == 2012].copy()
# Define quarters: Q2 = Apr(4),May(5),Jun(6); Q3 = Jul(7),Aug(8),Sep(9)
q2_mask = df2012['Month'].isin([4,5,6])
q3_mask = df2012['Month'].isin([7,8,9])
q2_totals = df2012[q2_mask].groupby('Store')['Weekly_Sales'].sum().rename('Q2_2012')
q3_totals = df2012[q3_mask].groupby('Store')['Weekly_Sales'].sum().rename('Q3_2012')
growth_df = pd.concat([q2_totals, q3_totals], axis=1).fillna(0)
# Compute growth; avoid division by zero by using NA where Q2==0
growth_df['Growth'] = (growth_df['Q3_2012'] - growth_df['Q2_2012']) / growth_df['Q2_2012'].replace(0, pd.NA)
growth_df = growth_df.sort_values('Growth', ascending=False)
# Show top 10 growers by growth
print('Top 10 stores by Q3 vs Q2 growth (2012):')
display(growth_df.head(10))
# Summary counts and lists
positive = growth_df[growth_df['Growth'] > 0]
print(f'Number of stores with positive growth: {positive.shape[0]}')
print('\nStores with growth > 10%:')
display(growth_df[growth_df['Growth'] > 0.10])
# Return the growth_df for further inspection if needed
growth_df

In [ ]:
# Basic EDA: types, missing values, summary
df.info()
print('
Missing values by column:
', df.isnull().sum())
display(df.describe(include='all'))

# Holiday weeks count
print('Holiday weeks:', df['Holiday_Flag'].value_counts().to_dict())

In [ ]:
# Feature engineering
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)

# Sort then create lag and rolling features per store
df = df.sort_values(['Store','Date']).reset_index(drop=True)
df['Weekly_Sales_lag1'] = df.groupby('Store')['Weekly_Sales'].shift(1)
df['Weekly_Sales_roll3'] = df.groupby('Store')['Weekly_Sales'].shift(1).rolling(3).mean().reset_index(level=0, drop=True)

# Simple fill for NaNs created by lagging (keep a note: better imputation possible)
df['Weekly_Sales_lag1'] = df['Weekly_Sales_lag1'].fillna(df['Weekly_Sales'].median())
df['Weekly_Sales_roll3'] = df['Weekly_Sales_roll3'].fillna(df['Weekly_Sales'].median())

# One-hot encode Store (45 stores ok for a baseline)
store_dummies = pd.get_dummies(df['Store'].astype(str), prefix='Store', drop_first=True)
df = pd.concat([df, store_dummies], axis=1)

# Quick check
df[['Store','Date','Weekly_Sales','Weekly_Sales_lag1','Weekly_Sales_roll3']].head()

In [ ]:
# Prepare features and target for baseline model
features = ['Temperature','Fuel_Price','CPI','Unemployment','Holiday_Flag','Year','Month','WeekOfYear','Weekly_Sales_lag1','Weekly_Sales_roll3'] + [c for c in df.columns if c.startswith('Store_')]
target = 'Weekly_Sales'
X = df[features].copy()
y = df[target].values
weights = np.where(df['Holiday_Flag']==1, 5.0, 1.0)

# Train/test split by time to avoid leakage: last ~12 weeks as test
cut_date = df['Date'].max() - pd.Timedelta(weeks=12)
train_idx = df['Date'] <= cut_date
X_train, X_test = X[train_idx], X[~train_idx]
y_train, y_test = y[train_idx], y[~train_idx]
w_train, w_test = weights[train_idx], weights[~train_idx]

print('Train rows:', X_train.shape[0], 'Test rows:', X_test.shape[0])

In [ ]:
# Weighted RMSE metric (holiday weeks weighted 5x as described)
def weighted_rmse(y_true, y_pred, sample_weight):
    mse = np.average((y_true - y_pred) ** 2, weights=sample_weight)
    return np.sqrt(mse)

# Baseline model: RandomForest
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
w_rmse = weighted_rmse(y_test, y_pred, w_test)
print(f'Plain RMSE: {rmse:.2f}')
print(f'Weighted RMSE (holiday 5x): {w_rmse:.2f}')

# Display a small comparison table
comp = pd.DataFrame({'Date': df.loc[~train_idx,'Date'], 'Store': df.loc[~train_idx,'Store'], 'Actual': y_test, 'Predicted': y_pred, 'Holiday_Flag': df.loc[~train_idx,'Holiday_Flag']})
comp.sort_values('Date').head(10)

**Next steps (suggested):**
- Tune the model (grid search / random search).
- Improve imputation for lag/rolling features and create additional lags.
- Model markdown events explicitly and add external features if available (e.g., promotions).
- Try store-level models or hierarchical approaches.
- Produce submission file and validate on holdout.

**How to run:** Use Jupyter Notebook or execute the notebook headlessly with `jupyter nbconvert --to notebook --execute Walmart_Project.ipynb`.